In [ ]:
#| default_exp mcp

## MCP server

Expose nbskill notebook operations as native MCP tools. This is the preferred integration for careful single-notebook reads and edits because multiline notebook cells travel as structured tool arguments rather than shell-quoted strings.

The server now favors verifiable context: batch edits include read-back hashes, symbol graph calls include complete structured usage data, and generated-file warnings are reserved for changes that touch exported notebook code.

The command-line functions are useful on their own, but coding agents work best when the same operations are available as structured tools. This notebook exposes the project through a FastMCP server while keeping the server layer thin and predictable.

### Production contract

The MCP server exposes the production core through stable structured tools. Tool schemas must hide CLI-only flags, responses must include concise text plus structured content, diagnostics must be scoped to touched notebooks when possible, edit tools must report feedback without raw notebook JSON, and experimental tools must be clearly marked or omitted from default production use.

The MCP server should stay boring on purpose. Each tool accepts structured arguments, captures printed output, uses notebook locks where file operations can collide, and delegates the actual work to the same functions tested elsewhere.

```python
mcp = create_mcp()
# MCP clients see tools such as project_context, file_context, chapter_context, symbol_context, write_nb, update_cell, exec_nb, and diff_nb.
```

In [ ]:
from contextlib import redirect_stdout
from io import StringIO
import nbskill.mcp as _mcp_mod
from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import write_nb as _write_raw_nb
from nbskill.mcp import capture_call as _example_capture_call
from nbskill.mcp import create_mcp as _example_create_mcp
from nbskill.read import file_context as _example_file_context
from nbskill.write import write_nb as _example_write_nb
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook

In [ ]:
def demo_tool():
    print("captured output")

print(_example_capture_call(demo_tool))
print(type(_example_create_mcp()).__name__)

In [ ]:
#| export
import hashlib,json,os
import re
import shutil
import subprocess
import sys
import threading
import time
from contextlib import redirect_stdout, redirect_stderr
from importlib.metadata import PackageNotFoundError, version
from io import StringIO
from pathlib import Path
from urllib.parse import unquote, urlparse

In [ ]:
#| export
from fastcore.nbio import read_nb as _read_raw_nb
from fastcore.script import _in_call_parse
from fastmcp import Context, FastMCP
from fastmcp.tools import ToolResult
from mcp.types import TextContent

In [ ]:
#| export
from nbskill.convert import new_nbdev_notebook
from nbskill.convert import py2nb
from nbskill.convert import py2nbdev
from nbskill.edit_interactive import execute_plan
from nbskill.edit_interactive import execute_project_plan
from nbskill.edit_interactive import plan_result_text
from nbskill.execute import exec_nb
from nbskill.workbench import agent_workbench_result
from nbskill.foundation import empty_failure_map, failure_map_path, load_failure_map
from nbskill.foundation import cell_source, exported_py_path, generated_owner, git_root, git_status_paths, notebook_paths, path_candidates

In [ ]:
#| export
from nbskill.graph import notebook_order_problems
from nbskill.graph import private_symbol_report
from nbskill.graph import symbol_graph_data, symbol_graph_public_data
from nbskill.graph import symbol_graph
from nbskill.knowledge import add_behaviour_steering
from nbskill.knowledge import get_knowledge
from nbskill.knowledge import reference_query
from nbskill.knowledge import store_knowledge
from nbskill.parallel import notebook_locks
from nbskill.read import chapter_context
from nbskill.read import file_context
from nbskill.read import project_context
from nbskill.read import symbol_context


In [ ]:
#| export
from nbskill.edit import edit_notebook
from nbskill.review import reset_global_usage_summary
from nbskill.review import notebook_size_problems
from nbskill.review import notebook_validation_problems
from nbskill.review import public_function_literacy_problems
from nbskill.review import run_style_check
from nbskill.review import style_check
from nbskill.review import style_report
from nbskill.review import diff_nb
from nbskill.write import apply_notebook_edit
from nbskill.write import insert_notebook_cells
from nbskill.write import replace_notebook_cell
from nbskill.write import replace_notebook_range
from nbskill.write import should_run_cell_feedback
from nbskill.write import source_lines_cells

### Capturing command output

The MCP tools should return text, not leak stdout and stderr into the server process. These helpers capture each underlying function call and convert its visible result into one response string.

In [ ]:
#| export
def as_text(value):
    return "" if value is None else str(value)

In [ ]:
#| export
_CAPTURE_LOCK = threading.RLock()

In [ ]:
#| export
_REDACT_KEYS = {"new", "new_lines", "source_lines", "replacement_lines", "cells", "edits", "operations", "source", "old_str", "new_str"}

In [ ]:
#| export
_REMOVED_SCRIPT_NAMES = (
    "nbskill-mcp", "read-nb", "write-nb", "update-cell", "batch-edit-nb",
    "show-doc", "exec-nb", "diff-nb", "style-check", "symbol-graph", "private-symbol-report",
)

In [ ]:
#| export
def _package_version(name="nbskill"):
    try: return version(name)
    except PackageNotFoundError: return "unknown"

In [ ]:
#| export
def capture_call(func, **kwargs):
    out, err = StringIO(), StringIO()
    with _CAPTURE_LOCK:
        original_stdout, original_stderr = sys.stdout, sys.stderr
        try:
            try:
                with redirect_stdout(out), redirect_stderr(err):
                    result = func(**kwargs)
            except SystemExit as exc:
                chunks = []
                if out.getvalue(): chunks.append(out.getvalue().rstrip())
                if err.getvalue(): chunks.append(err.getvalue().rstrip())
                chunks.append(f"SystemExit: {exc.code}")
                raise RuntimeError(chr(10).join(chunk for chunk in chunks if chunk)) from exc
        finally:
            sys.stdout, sys.stderr = original_stdout, original_stderr
    chunks = []
    if out.getvalue(): chunks.append(out.getvalue().rstrip())
    if err.getvalue(): chunks.append(err.getvalue().rstrip())
    if result is not None and not chunks: chunks.append(as_text(result))
    return chr(10).join(chunk for chunk in chunks if chunk)

In [ ]:
#| export
def capture_notebook_call(func, *paths, **kwargs):
    "Capture a call while holding per-notebook locks for `paths`."
    with notebook_locks(*paths):
        return capture_call(func, **kwargs)

In [ ]:
#| export
def _mcp_find_cell(path, cell_id):
    nb = _read_raw_nb(path)
    for cell in nb.cells:
        if getattr(cell, "id", None) == cell_id: return cell
    raise ValueError(f"Cell id {cell_id!r} was not found in {path}")

In [ ]:
#| export
def _mcp_source_hash(source):
    return hashlib.sha256(str(source).encode("utf-8")).hexdigest()[:12]

In [ ]:
#| export
def _mcp_cell_source_hash(path, cell_id):
    return _mcp_source_hash(cell_source(_mcp_find_cell(path, cell_id)))

In [ ]:
#| export
def _mcp_expected_hash_warning(path, cell_id, expected_hash):
    if not expected_hash: return None
    actual = _mcp_cell_source_hash(path, cell_id)
    if actual == expected_hash: return None
    return _warning(
        "expected_hash_mismatch",
        f"Skipped edit for {path} id={cell_id}: expected hash {expected_hash}, found {actual}.",
        "Refresh file_context or chapter_context and retry with the current source.",
        path=str(path), cell_id=cell_id, expected_hash=expected_hash, actual_hash=actual,
    )

In [ ]:
#| export
def _capture_exec_nb_cli_call(arguments):
    call_args = {key: value for key, value in arguments.items() if key not in {"detail", "workspace_root"}}
    script = "; ".join([
        "import json, sys",
        "from nbskill.execute import exec_nb",
        "exec_nb(**json.loads(sys.argv[1]))",
    ])
    lock_paths = [call_args.get("path")]
    if call_args.get("dest"): lock_paths.append(call_args["dest"])
    cwd = git_root(call_args.get("path") or ".") or _git_base(call_args.get("path") or ".").resolve()
    with notebook_locks(*lock_paths):
        proc = subprocess.run(
            [sys.executable, "-c", script, json.dumps(call_args)],
            text=True, capture_output=True, cwd=str(cwd),
        )
    chunks = [item.rstrip() for item in (proc.stdout, proc.stderr) if item]
    output = chr(10).join(chunks)
    if proc.returncode != 0:
        raise RuntimeError(output or f"exec_nb subprocess failed with exit code {proc.returncode}")
    return output

In [ ]:
#| export
def _mcp_cell_index(path, cell_id):
    nb = _read_raw_nb(path)
    for index, cell in enumerate(nb.cells):
        if getattr(cell, "id", None) == cell_id: return index
    raise ValueError(f"Cell id {cell_id!r} was not found in {path}")

In [ ]:
#| export
def _mcp_cell_ids_from_index(path, start, count):
    if start is None or count <= 0: return []
    nb = _read_raw_nb(path)
    return [
        getattr(cell, "id", None) for cell in nb.cells[start:start + count]
        if getattr(cell, "id", None)
    ]

In [ ]:
#| export
def _mcp_structured_cell_count(cells, default_cell_type="code"):
    return sum(len(source_lines_cells(cell, default_cell_type)) for cell in (cells or []))

In [ ]:
#| export
def _mcp_feedback_location(path, edit, default_cell_type="code"):
    edit_path = str(edit.get("path") or path)
    op = edit.get("op")
    if op == "replace_cell":
        cell_id = edit.get("cell_id")
        return edit_path, _mcp_cell_index(edit_path, cell_id), _mcp_structured_cell_count([edit], default_cell_type)
    if op == "replace_range":
        return edit_path, _mcp_cell_index(edit_path, edit.get("cell_id")), 1
    if op in {"insert_before", "insert_after"}:
        anchor_id = edit.get("anchor_id") or edit.get("cell_id")
        anchor_index = _mcp_cell_index(edit_path, anchor_id)
        start = anchor_index if op == "insert_before" else anchor_index + 1
        return edit_path, start, _mcp_structured_cell_count(edit.get("cells") or [edit], default_cell_type)
    return edit_path, None, 0

In [ ]:
#| export
def _mcp_feedback_output(path, cell_ids, auto_feedback=True, feedback_timeout=10, feedback_safe=True):
    if not auto_feedback: return ""
    nb = _read_raw_nb(path)
    cells_by_id = {getattr(cell, "id", None): cell for cell in nb.cells}
    target_id = None
    for cell_id in cell_ids:
        cell = cells_by_id.get(cell_id)
        if cell is not None and should_run_cell_feedback(cell): target_id = cell_id
    if target_id is None: return ""
    output = _capture_exec_nb_cli_call(dict(
        path=str(path), dest=None, exc_stop=False, up2id=target_id, chapter=None,
        timeout=feedback_timeout, show_output=True, verbose=False, safe=feedback_safe,
        allow=None, ok_dests=None, cache_httpx=False, cache_dir=None,
        cache_domains=None, allow_new=True, check_only=True,
    ))
    return f"Auto feedback (up to id={target_id}):\n{output}".rstrip()

In [ ]:
#| export
def _append_mcp_feedback(message, path, cell_ids, auto_feedback=True, feedback_timeout=10, feedback_safe=True):
    feedback = _mcp_feedback_output(path, cell_ids, auto_feedback, feedback_timeout, feedback_safe)
    return "\n\n".join(chunk for chunk in [message.rstrip(), feedback] if chunk)

In [ ]:
#| export
def _json_preview(value, limit=1200):
    text = json.dumps(value, indent=2, sort_keys=True, default=str)
    if len(text) <= limit: return text
    return f"{text[:limit].rstrip()}\n... truncated ..."

In [ ]:
#| export
def _text_preview(value, limit=12000):
    text = as_text(value)
    if limit is None or len(text) <= limit:
        return {"text": text, "truncated": False, "chars": len(text), "omitted_chars": 0}
    omitted = len(text) - limit
    return {
        "text": f"{text[:limit].rstrip()}\n... truncated {omitted} chars ...",
        "truncated": True,
        "chars": len(text),
        "omitted_chars": omitted,
    }

In [ ]:
#| export
def _redact_value(key, value, limit=160):
    if value is None: return None
    text = as_text(value)
    if key in _REDACT_KEYS and len(text) > limit:
        return f"<{len(text)} chars redacted; use detail='debug' to inspect>"
    if len(text) > limit * 3:
        return f"{text[:limit].rstrip()}... <{len(text) - limit} more chars>"
    return value

In [ ]:
#| export
def _redact_arguments(arguments):
    return {key: _redact_value(key, value) for key, value in (arguments or {}).items()}

In [ ]:
#| export
def _warning(code, message, next_action=None, **extra):
    item = {"code": code, "message": message}
    if next_action: item["next_action"] = next_action
    item.update({key: value for key, value in extra.items() if value is not None})
    return item

In [ ]:
#| export
def _path_without_cwd_prefix(path):
    return path_candidates(path)[-1]

In [ ]:
#| export
def _git_base(path="."):
    base = _path_without_cwd_prefix(path)
    if (base.exists() and not base.is_dir()) or (not base.exists() and base.suffix):
        return base.parent
    return base

In [ ]:
#| export
def _rooted_path(path, root):
    raw = Path(str(path)).expanduser()
    pth = _path_without_cwd_prefix(path)
    candidates = [pth]
    if root is not None and not raw.is_absolute():
        candidates = [Path(root) / raw, Path(root) / pth, *candidates]
    for candidate in candidates:
        try:
            if candidate.exists(): return candidate.resolve()
        except OSError:
            continue
    return pth.resolve()

In [ ]:
#| export
def _file_uri_path(uri):
    parsed = urlparse(str(uri))
    if parsed.scheme != "file": return None
    netloc = f"//{parsed.netloc}" if parsed.netloc else ""
    return Path(unquote(f"{netloc}{parsed.path}"))


async def _mcp_client_roots(ctx=None):
    if ctx is None: return []
    try: roots = await ctx.list_roots()
    except Exception: return []
    paths = []
    for root in roots:
        path = _file_uri_path(getattr(root, "uri", ""))
        if path is not None: paths.append(path)
    return paths


async def _mcp_workspace_root(ctx=None):
    for path in await _mcp_client_roots(ctx):
        try:
            if path.exists(): return path.resolve()
        except OSError:
            continue
    return None


def _mcp_workspace_path(path, root):
    if path in (None, ""): return path
    raw = Path(str(path)).expanduser()
    if raw.is_absolute() or root is None: return str(raw)
    return str((Path(root) / raw).resolve())


def _mcp_workspace_paths(paths, root):
    if paths in (None, ""): return paths
    return ",".join(_mcp_workspace_path(item.strip(), root) for item in str(paths).split(",") if item.strip())


def _mcp_workspace_edit_paths(edits, root):
    resolved = []
    for edit in edits:
        item = dict(edit)
        if item.get("path"): item["path"] = _mcp_workspace_path(item["path"], root)
        resolved.append(item)
    return resolved


def _mcp_workspace_call_args(arguments, root, *keys):
    data = dict(arguments)
    for key in keys:
        if key in data: data[key] = _mcp_workspace_path(data[key], root)
    return data

In [ ]:
#| export
def _rel_to_root(path, root):
    try: return _rooted_path(path, root).relative_to(Path(root).resolve()).as_posix()
    except (OSError, ValueError): return str(path)

In [ ]:
#| export
def _generated_files(root):
    skip = {".git", ".venv", "__pycache__", ".mypy_cache", ".pytest_cache"}
    items = []
    for path in Path(root).rglob("*.py"):
        if any(part in skip for part in path.parts): continue
        owner = generated_owner(path)
        if owner is not None: items.append((path, owner))
    return items

In [ ]:
#| export
def _owner_output(path):
    owner = generated_owner(path)
    if owner is None: return f"No generated-notebook owner found for {path}"
    return f"Generated file owner: {path} -> {owner}"

In [ ]:
#| export
def _failure_data():
    path = failure_map_path()
    try: return load_failure_map(path) if path.exists() else empty_failure_map()
    except OSError: return empty_failure_map()

In [ ]:
#| export
def _resolve_diagnostic_scope(path, root, scope_path=None):
    raw = path if scope_path in (None, "") else scope_path
    if raw in (None, "", "."): return Path(root)
    scoped = Path(raw).expanduser()
    if not scoped.is_absolute(): scoped = Path(root) / scoped
    return scoped

In [ ]:
#| export
def _is_project_scope(path, root):
    try: return Path(path).resolve() == Path(root).resolve()
    except OSError: return False

In [ ]:
#| export
def _generated_pairs_for_scope(root, path):
    path = Path(path)
    if _is_project_scope(path, root): return _generated_files(root)
    if path.suffix == ".py":
        owner = generated_owner(path)
        return [(path, owner)] if owner is not None else []
    pairs = []
    for nb_path in notebook_paths(path):
        try: py_path = exported_py_path(nb_path)
        except (FileNotFoundError, OSError): py_path = None
        if py_path is not None and Path(py_path).exists(): pairs.append((Path(py_path), nb_path.resolve()))
    return pairs

In [ ]:
#| export
def _cell_source_text(cell):
    if isinstance(cell, dict):
        source = cell.get("source", "")
        return "".join(source) if isinstance(source, list) else str(source)
    return cell_source(cell)

In [ ]:
#| export
def _cell_type_text(cell):
    return cell.get("cell_type", "") if isinstance(cell, dict) else getattr(cell, "cell_type", "")

In [ ]:
#| export
def _cell_export_relevant(cell):
    if _cell_type_text(cell) != "code": return False
    return any(re.match(r"^\s*#\|\s*(export|default_exp)\b", line) for line in _cell_source_text(cell).splitlines())

In [ ]:
#| export
def _notebook_export_relevant_change(owner, root):
    owner = Path(owner)
    root = Path(root)
    owner_rel = _rel_to_root(owner, root)
    proc = subprocess.run(["git", "-C", str(root), "show", f"HEAD:{owner_rel}"], text=True, capture_output=True)
    if proc.returncode != 0: return True
    try:
        old_nb = json.loads(proc.stdout)
        new_nb = json.loads(owner.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return True
    old_cells = {cell.get("id"): cell for cell in old_nb.get("cells", []) if cell.get("id")}
    new_cells = {cell.get("id"): cell for cell in new_nb.get("cells", []) if cell.get("id")}
    for cell_id in set(old_cells) | set(new_cells):
        old_cell, new_cell = old_cells.get(cell_id), new_cells.get(cell_id)
        if old_cell is None or new_cell is None:
            changed = True
        else:
            changed = _cell_type_text(old_cell) != _cell_type_text(new_cell) or _cell_source_text(old_cell) != _cell_source_text(new_cell)
        if changed and any(_cell_export_relevant(cell) for cell in (old_cell, new_cell) if cell is not None):
            return True
    return False

In [ ]:
#| export
def _style_problem_warnings(path, root):
    warnings = []
    problems = [*notebook_size_problems(path), *public_function_literacy_problems(path)]
    for problem in problems:
        code = problem.get("code")
        if code == "large-cell":
            cell = f" cell id={problem['cell_id']}" if problem.get("cell_id") else ""
            warnings.append(_warning(
                "large_cell",
                f"Notebook {_rel_to_root(problem.get('path'), root)}{cell} is large: {problem.get('detail')}.",
                "Split the cell so it contains one idea before continuing.",
                path=problem.get("path"), cell_id=problem.get("cell_id"), problem=problem,
            ))
        elif code == "large-generated-py":
            generated = problem.get("exported_py_path")
            warnings.append(_warning(
                "large_generated_py",
                f"Generated file {_rel_to_root(generated, root)} is getting large: {problem.get('detail')}.",
                "Use the notebook split tool to split the source notebook/module.",
                path=problem.get("path"), generated=generated, problem=problem,
            ))
        elif str(code).startswith("public-function-"):
            missing = problem.get("missing") or "one-line docstring"
            symbol = problem.get("symbol", "<unknown>")
            warnings.append(_warning(
                code.replace("-", "_"),
                f"Notebook {_rel_to_root(problem.get('path'), root)} public function {symbol!r} is missing {missing}.",
                "Add Markdown docs, a one-line docstring, an example cell, and a focused test cell.",
                path=problem.get("path"), cell_id=problem.get("cell_id"), problem=problem,
            ))
    return warnings

In [ ]:
#| export
def _warning_scope_root_rel(path, root):
    if path in (None, ""): return ""
    try:
        return Path(path).expanduser().resolve().relative_to(Path(root).resolve()).as_posix()
    except (OSError, ValueError):
        return str(path).strip()

In [ ]:
#| export
def _warning_scope_matches(value, root, scope):
    if scope is None: return True
    rel = _warning_scope_root_rel(value, root)
    return any(rel == item or rel.startswith(f"{item.rstrip('/')}/") for item in scope)

In [ ]:
#| export
def _warning_related_paths(item):
    for key in ("path", "generated", "owner"):
        if item.get(key): yield item[key]
    problem = item.get("problem") or {}
    for key in ("path", "exported_py_path"):
        if problem.get(key): yield problem[key]

In [ ]:
#| export
def _warning_cell_id(item):
    problem = item.get("problem") or {}
    return item.get("cell_id") or problem.get("cell_id")

In [ ]:
#| export
_NOTEBOOK_LEVEL_CONTEXT_WARNING_CODES = {
    "generated_without_notebook", "notebook_export_missing", "exported_py_hash_mismatch",
}

In [ ]:
#| export
def _filter_warnings_for_scope(warnings, root, scope_path=None, scope_cell_ids=None):
    if scope_path in (None, "", ".") and scope_cell_ids is None: return warnings
    scope = None if scope_path in (None, "", ".") else {_warning_scope_root_rel(scope_path, root).rstrip("/")}
    cell_filter_active = scope_cell_ids is not None
    cell_ids = set(scope_cell_ids or [])
    filtered = []
    for item in warnings:
        if scope is not None and not any(_warning_scope_matches(path, root, scope) for path in _warning_related_paths(item)):
            continue
        cell_id = _warning_cell_id(item)
        if cell_filter_active and cell_id and cell_id not in cell_ids: continue
        if cell_filter_active and not cell_id and item.get("code") not in _NOTEBOOK_LEVEL_CONTEXT_WARNING_CODES: continue
        filtered.append(item)
    return filtered

In [ ]:
#| export
def _doctor_warnings(path=".", scope_path=None, scope_cell_ids=None):
    root = git_root(path) or _git_base(path).resolve()
    diagnostic_path = _resolve_diagnostic_scope(path, root, scope_path)
    project_scope = _is_project_scope(diagnostic_path, root)
    changed = git_status_paths(root) if (root / ".git").exists() else set()
    warnings = []
    for py_path, owner in _generated_pairs_for_scope(root, diagnostic_path):
        py_rel = _rel_to_root(py_path, root)
        owner_rel = _rel_to_root(owner, root)
        if py_rel in changed and owner_rel not in changed:
            warnings.append(_warning(
                "generated_without_notebook",
                f"Generated file {py_rel} changed without its source notebook {owner_rel}.",
                "Move the edit into the notebook and export, or verify the generated edit is intentional.",
                path=py_rel, owner=owner_rel,
            ))
        if owner_rel in changed and py_rel not in changed:
            if _notebook_export_relevant_change(owner, root):
                warnings.append(_warning(
                    "notebook_export_missing",
                    f"Notebook {owner_rel} changed but generated file {py_rel} is unchanged.",
                    "Run export or use an nbskill write tool before shipping.",
                    path=owner_rel, generated=py_rel,
                ))
    for problem in notebook_validation_problems(diagnostic_path):
        if problem.get("code") != "exported-py-hash-mismatch": continue
        warnings.append(_warning(
            "exported_py_hash_mismatch",
            f"Notebook {problem['path']} metadata does not match current generated file {problem.get('exported_py_path')}.",
            "Run nbskill_validate or export the notebook through nbskill write tools.",
            path=problem.get("path"), generated=problem.get("exported_py_path"),
        ))
    warnings.extend(_style_problem_warnings(diagnostic_path, root))
    if project_scope:
        warnings.extend(_doc_script_warnings(root))
        failures = _failure_data().get("events", [])[-10:]
        recent_failures = [event for event in failures if event.get("kind") == "failure"]
        if recent_failures:
            last = recent_failures[-1]
            warnings.append(_warning(
                "recent_tool_failures",
                f"Recent nbskill failure: {last.get('tool')} {last.get('summary') or last.get('error')}",
                "Run doctor(detail='debug') or style_check(delete_after_output=True) after resolving it.",
                tool=last.get("tool"),
            ))
    if scope_path is None: scope_path = path
    return _filter_warnings_for_scope(warnings, root, scope_path, scope_cell_ids)

In [ ]:
#| export
def _doc_script_warnings(root):
    docs = [Path(root) / "README.md", Path(root) / "nbskill" / "SKILL.md"]
    docs += list((Path(root) / "nbskill" / "references").glob("*.md")) if (Path(root) / "nbskill" / "references").exists() else []
    warnings = []
    for doc in docs:
        if not doc.exists(): continue
        try: text = doc.read_text(encoding="utf-8", errors="ignore")
        except OSError: continue
        found = sorted(name for name in _REMOVED_SCRIPT_NAMES if name in text)
        if found:
            warnings.append(_warning(
                "removed_script_name",
                f"{doc.relative_to(root)} references removed CLI names: {', '.join(found)}.",
                "Replace hyphenated command names with underscore script names.",
                path=str(doc), names=found,
            ))
    return warnings

In [ ]:
#| export
def _first_match(pattern, text, flags=0):
    match = re.search(pattern, text or "", flags)
    return match.group(1) if match else None

In [ ]:
#| export
def _line_cell_ids(text):
    return set(re.findall(r"^Cell id=([^\s:]+)", text or "", re.MULTILINE))

In [ ]:
#| export
def _response_scope_cell_ids(tool, arguments, preview):
    text = preview.get("text", "")
    if tool in {"project_context", "file_context"}: return None
    if tool == "chapter_context": return _line_cell_ids(text) or None
    if tool == "symbol_context":
        cell_id = _first_match(r"^Location:.*?Cell id=([^\s:]+)", text, re.MULTILINE)
        return {cell_id} if cell_id else None
    if tool == "diff_nb":
        ids = set(re.findall(r"--- code cell ([^\s]+) ---", text))
        return ids if ids else set()
    if tool == "edit_notebook":
        ids = arguments.get("affected_cell_ids") or []
        if not ids and isinstance(arguments.get("edit_notebook"), dict):
            ids = arguments["edit_notebook"].get("affected_cell_ids", [])
        if not ids:
            ids = [str(item.get("cell_id") or item.get("anchor_id")) for item in arguments.get("edits", []) if isinstance(item, dict) and (item.get("cell_id") or item.get("anchor_id"))]
        return set(str(item) for item in ids if item) or None
    if tool == "exec_nb":
        up2id = arguments.get("up2id")
        if isinstance(up2id, str) and up2id and not up2id.isdigit(): return {up2id}
        return _line_cell_ids(text) or None
    for key in ("id", "cell_id", "any_cell_id"):
        value = arguments.get(key)
        if value: return {str(value)}
    return None

In [ ]:
#| export
def _unique_strings(items):
    seen, result = set(), []
    for item in items:
        if item in seen: continue
        seen.add(item)
        result.append(item)
    return result

In [ ]:
#| export
def _notebook_paths_from_text(text):
    return _unique_strings(re.findall(r"(?<![\w.-])(?:[~./A-Za-z0-9_-]+\.ipynb)", text or ""))

In [ ]:
#| export
def _edit_scope_paths(arguments):
    default = arguments.get("path")
    edits = arguments.get("edits") or []
    paths = [default] if default else []
    paths += [item.get("path") for item in edits if isinstance(item, dict) and item.get("path")]
    return _unique_strings(str(item) for item in paths if item)

In [ ]:
#| export
def _response_scope_paths(tool, arguments, preview):
    text_paths = _notebook_paths_from_text(preview.get("text", ""))
    if tool == "edit_notebook": return _unique_strings([*_edit_scope_paths(arguments), *text_paths])
    path = arguments.get("path") or arguments.get("nb_path")
    return [str(path)] if path else []

In [ ]:
#| export
def _dedupe_warnings(warnings):
    seen, result = set(), []
    for item in warnings:
        key = json.dumps(item, sort_keys=True, default=str)
        if key in seen: continue
        seen.add(key)
        result.append(item)
    return result

In [ ]:
#| export
def _response_warnings(tool, arguments, preview):
    warnings = []
    if preview.get("truncated"):
        warnings.append(_warning(
            "output_truncated",
            f"{tool} output was truncated by {preview['omitted_chars']} chars.",
            "Repeat with a narrower query or detail='debug' if you need full context.",
        ))
    read_context_tools = {"project_context", "file_context", "chapter_context", "symbol_context"}
    scoped_project_tools = {"exec_nb"}
    if tool in read_context_tools or tool == "edit_notebook": return _dedupe_warnings(warnings)
    if tool == "diff_nb":
        path = arguments.get("path") or "."
        cell_ids = _response_scope_cell_ids(tool, arguments, preview)
        warnings.extend(_doctor_warnings(path, scope_path=path, scope_cell_ids=cell_ids)[:3])
    elif tool in scoped_project_tools:
        cell_ids = _response_scope_cell_ids(tool, arguments, preview)
        for path in _response_scope_paths(tool, arguments, preview):
            warnings.extend(_doctor_warnings(path, scope_path=path, scope_cell_ids=cell_ids))
    return _dedupe_warnings(warnings)[:3]

In [ ]:
#| export
def _brief_call(tool, arguments, preview):
    lines = [f"{tool} completed"]
    for key in ("path", "nb_path", "cell_id", "id", "chapter", "name", "any_cell_id", "symbol"):
        if arguments.get(key) not in (None, ""):
            lines.append(f"{key}={arguments[key]}")
    if arguments.get("dry_run") is True: lines.append("dry_run=True")
    if preview.get("truncated"): lines.append(f"output_truncated=True omitted_chars={preview['omitted_chars']}")
    return lines

In [ ]:
#| export
def mcp_tool_result(tool, arguments, full_output, max_output_chars=12000, detail="summary", warnings=None, hints=None, **structured):
    "Return concise visible MCP text plus structured data for clients that inspect it."
    detail = detail or "summary"
    if detail == "debug": max_output_chars = max(max_output_chars, 50000)
    preview = _text_preview(full_output or "", limit=max_output_chars)
    warnings = _dedupe_warnings([*(warnings or []), *_response_warnings(tool, arguments or {}, preview)])
    hints = list(hints or [])
    lines = _brief_call(tool, arguments or {}, preview)
    if preview["text"]:
        lines += ["", "Result:", preview["text"]]
    if warnings:
        lines += ["", "Warnings:"]
        lines.extend(f"- {item['message']}" + (f" Next: {item['next_action']}" if item.get("next_action") else "") for item in warnings)
    if detail == "debug" and hints:
        lines += ["", "Hints:"]
        lines.extend(f"- {hint}" for hint in hints)
    summary = chr(10).join(lines)
    data = {
        "summary": summary,
        "call": {"tool": tool, "arguments": _redact_arguments(arguments or {})},
        "full_output": preview["text"],
        "output_truncated": preview["truncated"],
        "output_chars": preview["chars"],
        "omitted_chars": preview["omitted_chars"],
        "warnings": warnings,
        "hints": hints if detail == "debug" else [],
    }
    if detail == "debug":
        data["debug"] = {"arguments": arguments or {}, "raw_output": as_text(full_output or "")}
    data.update(structured)
    return ToolResult(content=[TextContent(type="text", text=summary)], structured_content=data)


In [ ]:
#| export
def _status_data(client_roots=None):
    scripts = [
        "project_context", "file_context", "chapter_context", "symbol_context",
        "write_nb", "update_cell", "batch_edit_nb", "exec_nb", "diff_nb", "style_check",
        "install_nbskill", "symbol_graph", "private_symbol_report", "agent_workbench",
        "new_nbdev_notebook", "add_behaviour_steering", "store_knowledge", "get_knowledge",
        "reference_add", "reference_list", "reference_ingest", "reference_query",
        "nbskill_mcp",
    ]
    client_roots = [str(Path(item).resolve()) for item in (client_roots or [])]
    workspace_root = client_roots[0] if client_roots else str(Path.cwd())
    return {
        "version": _package_version(),
        "cwd": str(Path.cwd()),
        "workspace_root": workspace_root,
        "client_roots": client_roots,
        "python": sys.executable,
        "mcp_command": "nbskill_mcp",
        "mcp_command_path": shutil.which("nbskill_mcp"),
        "cli_tools": {name: shutil.which(name) for name in scripts},
        "reconnect_hint": "Restart or reconnect the MCP client after reinstalling nbskill or changing tool signatures.",
        "install_commands": [
            "uv tool install --editable . --force",
            "codex mcp add nbskill -- nbskill_mcp",
            "claude mcp add nbskill -- nbskill_mcp",
        ],
    }

In [ ]:
#| export
def _doctor_report(
    path=".",
    detail="summary",
    fix=False,
    reset=False,
    capabilities="",
    scopes="error,warning",
    skip_folder_re=None,
    skip_path=None,
    max_output_chars=12000,
    max_diagnostics=200,
):
    root = git_root(path) or _git_base(path).resolve()
    status = _status_data()
    selected = _doctor_scope_set(scopes)
    errors = _doctor_error_items(path, status) if "error" in selected else []
    warnings, private_text = _doctor_warning_items(path) if "warning" in selected else ([], "")
    style = (
        _doctor_style_report(path, skip_folder_re=skip_folder_re, skip_path=skip_path, max_output_chars=max_output_chars, max_diagnostics=max_diagnostics)
        if "style" in selected else None
    )
    failure_map = _failure_data()
    if reset: reset_global_usage_summary()
    hints = [
        "Use scopes='error,warning,style' or scopes='all' for the full doctor report.",
        "Use scopes='style' to include chkstyle output; chkstyle is omitted from error/warning scopes.",
        "Use project_context/file_context/chapter_context/symbol_context, then edit_notebook for deterministic notebook mutations.",
    ]
    changed = sorted(git_status_paths(root)) if (root / ".git").exists() else []
    generated = [
        {"path": _rel_to_root(py, root), "owner": _rel_to_root(owner, root)}
        for py, owner in _generated_files(root)
    ]
    issue_count = len(errors) + len(warnings)
    style_count = (style or {}).get("summary", {}).get("diagnostic_count", 0)
    summary = f"nbskill doctor: {len(errors)} error(s), {len(warnings)} warning(s)"
    if "style" in selected: summary += f", {style_count} style diagnostic(s)"
    if not issue_count and "style" not in selected: summary = "nbskill doctor: no actionable errors or warnings"
    text_lines = [summary]
    if errors:
        text_lines.append("\nErrors:")
        text_lines.extend(f"- {item['message']}" + (f" Next: {item['next_action']}" if item.get("next_action") else "") for item in errors)
    if warnings:
        text_lines.append("\nWarnings:")
        text_lines.extend(f"- {item['message']}" + (f" Next: {item['next_action']}" if item.get("next_action") else "") for item in warnings)
    if style is not None and style.get("text", "").strip():
        text_lines.append("\nStyle:")
        text_lines.append(style["text"].strip())
    report = {
        "path": str(path),
        "root": str(root),
        "scopes": sorted(selected),
        "status": status,
        "errors": errors,
        "warnings": warnings,
        "issues": [*errors, *warnings],
        "style": style,
        "private_symbol_report": private_text if "warning" in selected else "",
        "hints": hints,
        "capabilities": [item for item in _MCP_CAPABILITIES.split(",") if item],
        "changed_paths": changed if detail == "debug" else changed[:20],
        "generated_owners": generated if detail == "debug" else generated[:20],
        "recent_events": failure_map.get("events", [])[-20:] if detail == "debug" else [],
        "counts": failure_map.get("counts", {}),
        "reset": bool(reset),
        "fix": {"requested": bool(fix), "applied": []},
        "text": "\n".join(text_lines),
    }
    return report

In [ ]:
#| export
def nbskill_status(json_output: bool = False):  # Keep argument for compatibility with the CLI wrapper
    "Report nbskill version, MCP command setup, canonical CLI tools, and reconnect hints."
    return _status_data()

In [ ]:
#| export
_DOCTOR_SCOPES = {"error", "warning", "style"}

In [ ]:
#| export
def _doctor_scope_set(scopes="error,warning"):
    "Normalize comma/space-separated doctor scopes."
    if scopes is None: return {"error", "warning"}
    raw = str(scopes).replace(",", " ").split()
    selected = set(raw) or {"error", "warning"}
    if "all" in selected: return set(_DOCTOR_SCOPES)
    unknown = selected - _DOCTOR_SCOPES
    if unknown: raise ValueError(f"Unknown doctor scope(s): {', '.join(sorted(unknown))}")
    return selected

In [ ]:
#| export
def _problem_message(problem):
    parts = [str(problem.get("path") or "")]
    if problem.get("cell_id"): parts.append(f"id={problem['cell_id']}")
    if problem.get("line"): parts.append(f"line={problem['line']}")
    if problem.get("symbol"): parts.append(f"symbol={problem['symbol']!r}")
    if problem.get("code"): parts.append(f"code={problem['code']}")
    if problem.get("detail"): parts.append(str(problem["detail"]))
    return " ".join(part for part in parts if part)

In [ ]:
#| export
def _doctor_validation_errors(path):
    errors = []
    for problem in notebook_validation_problems(path):
        errors.append(_warning(
            problem.get("code", "notebook-validation"),
            _problem_message(problem),
            "Fix notebook metadata/source ordering before relying on notebook edits.",
            severity="error", scope="error", problem=problem,
        ))
    return errors

In [ ]:
#| export
def _doctor_order_errors(path):
    errors = []
    try:
        problems = notebook_order_problems(path)
    except FileNotFoundError as exc:
        return [_warning(
            "notebook_order_missing_file",
            f"Notebook order scan found a missing notebook: {exc.filename or exc}",
            "Remove stale notebook references or recreate the missing notebook before rerunning doctor.",
            severity="error", scope="error", path=exc.filename,
        )]
    for problem in problems:
        errors.append(_warning(
            problem.get("code", "notebook-order"),
            _problem_message(problem),
            "Move definitions/imports before use, or add the missing import.",
            severity="error", scope="error", problem=problem,
        ))
    return errors

In [ ]:
#| export
def _doctor_recent_failure_errors():
    errors = []
    recent = [event for event in _failure_data().get("events", [])[-10:] if event.get("kind") == "failure"]
    if recent:
        last = recent[-1]
        errors.append(_warning(
            "recent_tool_failures",
            f"Recent nbskill failure: {last.get('tool')} {last.get('summary') or last.get('error')}",
            "Run doctor(detail='debug', scopes='error') after resolving it.",
            severity="error", scope="error", tool=last.get("tool"),
        ))
    return errors

In [ ]:
#| export
def _doctor_error_items(path, status):
    errors = [
        *_doctor_validation_errors(path),
        *_doctor_order_errors(path),
        *_doctor_recent_failure_errors(),
    ]
    if not status.get("mcp_command_path"):
        errors.append(_warning(
            "mcp_command_missing",
            "nbskill_mcp is not on PATH for this process.",
            "Run uv tool install --editable . --force and reconnect the MCP client.",
            severity="error", scope="error",
        ))
    return errors

In [ ]:
#| export
def _private_symbol_report_text(path):
    return capture_call(private_symbol_report, path=str(path))

In [ ]:
#| export
def _private_symbol_warnings(path):
    try:
        text = _private_symbol_report_text(path)
    except FileNotFoundError as exc:
        text = f"Private symbol scan found a missing notebook: {exc.filename or exc}"
        return [_warning(
            "private_symbol_missing_file",
            text,
            "Remove stale notebook references or recreate the missing notebook before rerunning doctor.",
            severity="warning", scope="warning", path=exc.filename,
        )], text
    if "No cross-notebook private symbol calls found." in text:
        return [], text
    warnings = [
        _warning(
            "private_symbol_call",
            line[2:],
            "Promote the helper to public API or keep the call inside the defining notebook.",
            severity="warning", scope="warning",
        )
        for line in text.splitlines()
        if line.startswith("- ")
    ]
    return warnings, text

In [ ]:
#| export
def _doctor_warning_items(path):
    error_codes = {"exported_py_hash_mismatch", "recent_tool_failures"}
    warnings = [
        {**item, "severity": item.get("severity", "warning"), "scope": "warning"}
        for item in _doctor_warnings(path)
        if item.get("code") not in error_codes
    ]
    private_warnings, private_text = _private_symbol_warnings(path)
    return [*warnings, *private_warnings], private_text

In [ ]:
#| export
def _doctor_style_report(path, skip_folder_re=None, skip_path=None, max_output_chars=12000, max_diagnostics=200):
    chkstyle = run_style_check(path, skip_folder_re, skip_path, strict=False, max_output_chars=max_output_chars)
    try:
        return style_report(
            path, chkstyle=chkstyle, max_output_chars=max_output_chars,
            max_diagnostics=max_diagnostics, skip_folder_re=skip_folder_re, skip_path=skip_path,
        )
    except FileNotFoundError as exc:
        message = f"Style scan found a missing notebook: {exc.filename or exc}"
        diagnostic = {
            "source": "notebook",
            "code": "style_missing_file",
            "severity": "error",
            "path": exc.filename,
            "detail": message,
        }
        output = chkstyle.get("output", "") if isinstance(chkstyle, dict) else ""
        truncated = max_output_chars is not None and len(output) > max_output_chars
        text = output[:max_output_chars].rstrip() if truncated else output.strip()
        if text: text = f"{text}\n\n{message}"
        else: text = message
        return {
            "path": str(path),
            "summary": {
                "notebook_problem_count": 1,
                "chkstyle_problem_count": 0,
                "diagnostic_count": 1,
                "recent_problem_count": 0,
                "output_truncated": truncated,
                "output_chars": len(output),
                "omitted_chars": max(0, len(output) - (max_output_chars or len(output))),
            },
            "diagnostics": [diagnostic],
            "problem_chart": {
                "by_code": {"style_missing_file": 1},
                "by_severity": {"error": 1},
                "by_path": {str(exc.filename): 1} if exc.filename else {},
                "by_source": {"notebook": 1},
            },
            "notebook_problems": [diagnostic],
            "global_usage": {},
            "chkstyle": {"status": chkstyle.get("status", 0) if isinstance(chkstyle, dict) else 0, "text": output[:max_output_chars] if truncated else output, "truncated": truncated, "chars": len(output), "omitted_chars": max(0, len(output) - (max_output_chars or len(output)))},
            "fixes": [],
            "text": text,
        }

In [ ]:
data = _status_data()
assert data["mcp_command"] == "nbskill_mcp"
assert "batch_edit_nb" in data["cli_tools"]

### Registering notebook tools

`create_mcp` is the bridge between this package and an agent client. Each tool is a thin wrapper around a public function, with notebook locks around operations that touch shared files.

The wrapper should preserve useful structure, not flatten everything into text. For example, `symbol_graph` still prints a readable summary, but MCP clients also receive a full `symbol_graph` payload they can inspect without parsing prose.

Edit and execution wrappers also adapt to MCP-specific transport. `update_cell` accepts `new_lines` so exact source can be sent directly without temporary files, and unsafe `exec_nb` runs through a subprocess because the underlying timeout machinery needs the main interpreter thread.

In [ ]:
#| export
_MCP_DIAGNOSTIC_TOOL_CATALOG = {
    'healthcheck': {
        'feature': 'diagnostics',
        'usefulness': 'core',
        'tags': ('status', 'diagnostics', 'setup'),
        'description': 'Cheap liveness probe for the nbskill MCP server, installed version, capabilities, concurrency policy, and reconnect hints.',
        'when_to_use': 'Call first when checking that the MCP server is connected or after reinstalling/exporting tool signatures.',
        'combine_with': 'Could be folded into doctor, but a cheap health probe is useful enough to keep separate.',
    },
    'doctor': {
        'feature': 'diagnostics',
        'usefulness': 'core',
        'tags': ('status', 'diagnostics', 'error', 'warning', 'style'),
        'description': 'Scoped diagnostics for MCP setup, fatal notebook problems, warnings, private symbol leaks, and optional chkstyle output.',
        'when_to_use': "Use scopes='error', scopes='warning', scopes='style', or scopes='all' depending on the diagnostic depth needed.",
        'combine_with': 'Now absorbs private symbol warnings and scoped style diagnostics; healthcheck stays separate as a cheap probe.',
    },
}

In [ ]:
#| export
_MCP_READ_CONTEXT_TOOL_CATALOG = {
    'project_context': {
        'feature': 'read_context',
        'usefulness': 'core',
        'tags': ('read', 'project', 'orientation'),
        'description': 'Project context with relevant README sections, notebook filenames, and notebook file docstrings.',
        'when_to_use': 'Start here when opening a repository or planning work across multiple notebooks.',
        'combine_with': 'Follow with file_context for one notebook or symbol_context for one implementation.',
    },
    'file_context': {
        'feature': 'read_context',
        'usefulness': 'core',
        'tags': ('read', 'notebook', 'file'),
        'description': 'Notebook file context with imports, header docs, Markdown cells, and function/class/method docstrings; supports include/exclude regex filters.',
        'when_to_use': 'Use before changing or reviewing one notebook file.',
        'combine_with': 'Use chapter_context for a section-level view or symbol_context for implementation detail.',
    },
    'chapter_context': {
        'feature': 'read_context',
        'usefulness': 'core',
        'tags': ('read', 'notebook', 'chapter'),
        'description': 'Notebook head plus one selected chapter found by chapter name, text query, or any cell id inside the chapter.',
        'when_to_use': 'Use when a section-level view is enough and whole-file context is too broad.',
        'combine_with': 'Use file_context first for file orientation or symbol_context when changing a concrete implementation.',
    },
    'symbol_context': {
        'feature': 'read_context',
        'usefulness': 'core',
        'tags': ('read', 'symbol', 'implementation'),
        'description': 'Implementation context for one symbol with exact source, mentioning Markdown, examples/tests with outputs, callers, and depth-controlled callees.',
        'when_to_use': 'Use before changing a function, class, or method, or when tracing how a symbol is used.',
        'combine_with': 'Use depth=0 for direct implementation context; increase depth to inspect callees under the hood.',
    },
}


In [ ]:
#| export
_MCP_READ_DOC_TOOL_CATALOG = {}


In [ ]:
#| export
_MCP_CELL_EDIT_TOOL_CATALOG = {
    'edit_notebook': {
        'feature': 'notebook_edit',
        'usefulness': 'core',
        'tags': ('edit', 'notebook', 'cell', 'text', 'batch'),
        'description': 'Apply deterministic notebook edit operations atomically across cells, lines, and notebook-wide text replacements; returns structured match counts, hashes, affected cell ids, diffs, warnings, and optional feedback.',
        'when_to_use': "Use after file_context, chapter_context, or symbol_context identifies target cells. Use replace_text/replace_texts with target='all' for notebook-level renames, line ops for focused cell edits, and structural ops for insert/delete/move/replace cell changes.",
        'combine_with': 'Read context before editing; review with diff_nb, exec_nb(check_only=True), doctor, or style_check afterwards.',
    },
}

In [ ]:
#| export
_MCP_BATCH_EDIT_TOOL_CATALOG = {}

In [ ]:
#| export
_MCP_REVIEW_TOOL_CATALOG = {
    'exec_nb': {
        'feature': 'verification',
        'usefulness': 'core',
        'tags': ('execute', 'notebook', 'verify', 'safe'),
        'description': 'Execute a notebook, chapter, or cells up to an id with safe-mode controls and visible output/error capture; unsafe execution is routed through a CLI subprocess so signal-based timeouts run in a main interpreter without blocking MCP worker threads.',
        'when_to_use': 'Use after edits or before trusting notebook behavior; use check_only=True when outputs should not be written.',
        'combine_with': 'Keep separate because execution has distinct safety and concurrency semantics.',
    },
    'diff_nb': {
        'feature': 'review',
        'usefulness': 'core',
        'tags': ('review', 'notebook', 'diff'),
        'description': 'Notebook-aware code-cell diff that avoids raw .ipynb noise and can map generated Python diffs back to notebook owners.',
        'when_to_use': 'Use before final reporting or when reviewing notebook edits without expanding JSON metadata churn.',
        'combine_with': 'Could be grouped with style_check under review, but diff parameters and output are meaningfully different.',
    },
    'style_check': {
        'feature': 'review',
        'usefulness': 'core',
        'tags': ('review', 'style', 'hygiene', 'privacy'),
        'description': 'Notebook hygiene and style report including chkstyle output, private symbol warnings, duplicate imports, stored knowledge warnings, and order issues.',
        'when_to_use': 'Use after substantial edits or when a notebook feels structurally messy; stored behaviour steering regexes are included as warnings.',
        'combine_with': "Doctor can include style diagnostics with scopes='style'; standalone style_check remains the explicit review tool.",
    },
}

In [ ]:
#| export
_MCP_AGENT_TOOL_CATALOG = {
    'execute_plan': {
        'feature': 'agentic_planning',
        'usefulness': 'advanced',
        'tags': ('agent', 'edit', 'plan', 'notebook', 'project'),
        'description': 'Run a bounded edit-interactive plan against one notebook or a project-scoped set of notebooks.',
        'when_to_use': "Use scope='notebook' with notebook=... for one notebook, or scope='project' with notebooks=... for broad plans.",
        'combine_with': 'Combined former execute_project_plan into this tool via scope.',
    },
    'agent_workbench': {
        'feature': 'agentic_planning',
        'usefulness': 'advanced',
        'tags': ('agent', 'context', 'taste', 'contract', 'review'),
        'description': 'Prepare or execute a taste-aware small-diff workbench run with context, budgets, and gates.',
        'when_to_use': 'Use before autonomous implementation when taste, scope, context, and patch budgets need to be explicit.',
        'combine_with': 'Sits above execute_plan; execute_plan remains the bounded notebook executor.',
    },
    'symbol_graph': {
        'feature': 'symbol_analysis',
        'usefulness': 'situational',
        'tags': ('analysis', 'symbol', 'graph'),
        'description': "Analyze one symbol's definitions, callers, caller usage lines, and callees across notebooks.",
        'when_to_use': 'Use when understanding impact, dependencies, or call relationships around one symbol; request JSON/structured content for migration scripts.',
        'combine_with': "Private symbol reporting moved into doctor(scope='warning') and style_check output.",
    },
}

In [ ]:
#| export
_MCP_KNOWLEDGE_TOOL_CATALOG = {
    'store_knowledge': {
        'feature': 'knowledge',
        'usefulness': 'core',
        'tags': ('knowledge', 'memory', 'style', 'regex'),
        'description': 'Store or update one behaviour steering regex and note in the nbskill JSON memory file.',
        'when_to_use': 'Use after translating a remembered good or bad practice into a regex that should warn in future style checks.',
        'combine_with': 'add_behaviour_steering is a shorter regex-only helper; get_knowledge reads stored rules.',
    },
    'add_behaviour_steering': {
        'feature': 'knowledge',
        'usefulness': 'situational',
        'tags': ('knowledge', 'memory', 'regex'),
        'description': 'Add a behaviour steering regex with a generic note.',
        'when_to_use': 'Use when the regex itself is enough context, or prefer store_knowledge when a human-readable note matters.',
        'combine_with': 'store_knowledge is the richer form and style_check applies both kinds of stored rules.',
    },
    'get_knowledge': {
        'feature': 'knowledge',
        'usefulness': 'core',
        'tags': ('knowledge', 'memory', 'lookup'),
        'description': 'Return stored behaviour steering rules, optionally filtered by regex.',
        'when_to_use': 'Use before adding a similar rule, or when inspecting why style_check emitted a stored knowledge warning.',
        'combine_with': 'Use store_knowledge to add or update rules.',
    },
    'reference_query': {
        'feature': 'reference_knowledge',
        'usefulness': 'core',
        'tags': ('knowledge', 'reference', 'search', 'implementation'),
        'description': 'Search globally indexed reference implementations and optionally include direct same-repo callers and callees.',
        'when_to_use': 'Use when a task needs examples from the user reference knowledgebase before implementing similar code.',
        'combine_with': 'CLI reference_add/reference_ingest build the index; MCP only exposes querying in v1.',
    },
}

In [ ]:
#| export
_MCP_CONVERT_TOOL_CATALOG = {
    'py2nb': {
        'feature': 'conversion',
        'usefulness': 'situational',
        'tags': ('convert', 'python', 'notebook', 'folder'),
        'description': 'Convert one Python file or a folder of Python files into nbdev notebook source with pragmatic cell splitting.',
        'when_to_use': 'Use for file-level or folder-level Python-to-notebook migration.',
        'combine_with': 'Combined former py2nbs behavior into this file-or-folder converter.',
    },
    'py2nbdev': {
        'feature': 'conversion',
        'usefulness': 'situational',
        'tags': ('convert', 'project', 'nbdev'),
        'description': 'Create a pragmatic nbdev project from a pure-Python package or project tree.',
        'when_to_use': 'Use when bootstrapping a whole nbdev project rather than converting one module or folder.',
        'combine_with': 'Keep separate from py2nb because it creates project structure, not only notebooks.',
    },
    'new_nbdev_notebook': {
        'feature': 'conversion',
        'usefulness': 'situational',
        'tags': ('create', 'notebook', 'nbdev'),
        'description': 'Create a minimal nbdev source notebook and exported module.',
        'when_to_use': 'Use when adding a new nbdev notebook/module to a project.',
        'combine_with': 'Use py2nb when converting existing Python source instead of starting a blank notebook.',
    },
}

In [ ]:
#| export
_MCP_TOOL_CATALOG = {
    **_MCP_DIAGNOSTIC_TOOL_CATALOG,
    **_MCP_READ_CONTEXT_TOOL_CATALOG,
    **_MCP_READ_DOC_TOOL_CATALOG,
    **_MCP_CELL_EDIT_TOOL_CATALOG,
    **_MCP_BATCH_EDIT_TOOL_CATALOG,
    **_MCP_REVIEW_TOOL_CATALOG,
    **_MCP_AGENT_TOOL_CATALOG,
    **_MCP_KNOWLEDGE_TOOL_CATALOG,
    **_MCP_CONVERT_TOOL_CATALOG,
}

In [ ]:
#| export
def _mcp_tool_meta(name):
    "Return FastMCP registration metadata for one nbskill tool."
    info = _MCP_TOOL_CATALOG[name]
    return {
        "name": name,
        "description": info["description"],
        "tags": set(info["tags"]),
        "meta": {
            "feature": info["feature"],
            "usefulness": info["usefulness"],
            "when_to_use": info["when_to_use"],
            "combine_with": info["combine_with"],
        },
    }

In [ ]:
#| export
_MCP_CAPABILITIES = ",".join(_MCP_TOOL_CATALOG)
mcp = FastMCP(
    "nbskill",
    instructions=(
        "Work notebook-first in nbdev projects. Feature areas are diagnostics, focused context, "
        "notebook edits, verification/review, symbol analysis, agentic planning, and conversion. "
        "For reading, use project_context for repository orientation, file_context for one notebook, "
        "chapter_context for one section, and symbol_context for a concrete implementation. "
        "For edits, use edit_notebook as the single production mutation tool. It supports whole-cell, "
        "line-range, insert/delete/move, and notebook-wide replace_text/replace_texts operations with "
        "expected_hash guards and structured diffs. Edit tools default to auto_feedback=True, which runs "
        "only exploratory or test cells in check-only mode and returns their output/errors. Use exec_nb, "
        "diff_nb, and style_check for verification and review; use doctor with scopes='error', 'warning', "
        "'style', or 'all' for diagnostics. Chkstyle output only appears when doctor includes the style "
        "scope. Reserve execute_plan for agentic notebook/project edits. Use store_knowledge/get_knowledge "
        "for behaviour steering memory; style_check applies stored regex rules as warnings. Normal tool "
        "output is concise; use detail='debug' only when troubleshooting. Notebook operations are "
        "concurrency-safe: calls touching the same notebook are serialized, calls touching different "
        "notebooks can run in parallel, and execution uses a global semaphore. Keep documentation before "
        "exported code and show-off examples after it."
    ),
)

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("healthcheck"))
async def _healthcheck_tool(detail: str = "summary", ctx: Context = None) -> ToolResult:
    "Return lightweight nbskill MCP status and point deeper diagnostics to doctor."
    client_roots = await _mcp_client_roots(ctx)
    data = _status_data(client_roots)
    full_output = "\n".join([
        "nbskill mcp ok",
        f"version={data['version']}",
        f"cwd={Path.cwd()}",
        f"workspace_root={data['workspace_root']}",
        f"python={sys.executable}",
        f"pid={os.getpid()}",
        f"capabilities={_MCP_CAPABILITIES}",
        "parallel=same-notebook operations serialized; different notebooks may run in parallel",
        "execution=global semaphore with one active safe notebook execution",
        "diagnostics=run doctor(scopes='error,warning') for fatal problems and warnings; add style for chkstyle",
        "schema_refresh=restart or reconnect the MCP client after reinstall/export to refresh tool schemas",
    ])
    return mcp_tool_result("healthcheck", {"detail": detail}, full_output, detail=detail, status=data, capabilities=_MCP_CAPABILITIES.split(","))

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("doctor"))
async def doctor_tool(
    path: str = ".", scopes: str = "error,warning", detail: str = "summary",
    fix: bool = False, reset: bool = False, skip_folder_re: str | None = None,
    skip_path: str | None = None, max_output_chars: int = 12000,
    max_diagnostics: int = 200, ctx: Context = None,
) -> ToolResult:
    "Report scoped MCP diagnostics: errors, warnings, and optional chkstyle/style details."
    root = await _mcp_workspace_root(ctx)
    path = _mcp_workspace_path(path, root)
    skip_path = _mcp_workspace_path(skip_path, root) if skip_path else skip_path
    arguments = dict(path=path, scopes=scopes, detail=detail, fix=fix, reset=reset, skip_folder_re=skip_folder_re, skip_path=skip_path, max_output_chars=max_output_chars, max_diagnostics=max_diagnostics, workspace_root=str(root) if root else None)
    report = _doctor_report(path=path, detail=detail, fix=fix, reset=reset, capabilities=_MCP_CAPABILITIES, scopes=scopes, skip_folder_re=skip_folder_re, skip_path=skip_path, max_output_chars=max_output_chars, max_diagnostics=max_diagnostics)
    return mcp_tool_result(
        "doctor", arguments, report["text"], detail=detail,
        warnings=report["issues"], hints=report["hints"], doctor=report,
    )

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("project_context"))
async def _project_context_tool(path: str = ".", detail: str = "summary", ctx: Context = None) -> ToolResult:
    "Show project README sections, notebook filenames, and notebook file docstrings."
    root = await _mcp_workspace_root(ctx)
    path = _mcp_workspace_path(path, root)
    arguments = dict(path=path, detail=detail, workspace_root=str(root) if root else None)
    with notebook_locks(path):
        data = project_context(path, verbose=False)
    return mcp_tool_result("project_context", arguments, data["text"], detail=detail, context=data)

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("chapter_context"))
async def _chapter_context_tool(path: str, query: str | None = None, name: str | None = None, any_cell_id: str | None = None, detail: str = "summary", ctx: Context = None) -> ToolResult:
    "Show the notebook head plus one selected chapter."
    root = await _mcp_workspace_root(ctx)
    path = _mcp_workspace_path(path, root)
    arguments = dict(path=path, query=query, name=name, any_cell_id=any_cell_id, detail=detail, workspace_root=str(root) if root else None)
    with notebook_locks(path):
        data = chapter_context(path, query=query, name=name, any_cell_id=any_cell_id, verbose=False)
    return mcp_tool_result("chapter_context", arguments, data["text"], detail=detail, context=data)

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("file_context"))
async def _file_context_tool(path: str, include_re: str | None = None, exclude_re: str | None = None, detail: str = "summary", ctx: Context = None) -> ToolResult:
    "Show one notebook's imports, docs, Markdown, and definitions with optional regex filters."
    root = await _mcp_workspace_root(ctx)
    path = _mcp_workspace_path(path, root)
    arguments = dict(path=path, include_re=include_re, exclude_re=exclude_re, detail=detail, workspace_root=str(root) if root else None)
    with notebook_locks(path):
        data = file_context(path, include_re=include_re, exclude_re=exclude_re, verbose=False)
    return mcp_tool_result("file_context", arguments, data["text"], detail=detail, context=data)

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("symbol_context"))
async def _symbol_context_tool(path: str, symbol: str, depth: int = 1, detail: str = "summary", ctx: Context = None) -> ToolResult:
    "Show exact implementation context, examples/tests, callers, and callees for a symbol."
    root = await _mcp_workspace_root(ctx)
    path = _mcp_workspace_path(path, root)
    arguments = dict(path=path, symbol=symbol, depth=depth, detail=detail, workspace_root=str(root) if root else None)
    with notebook_locks(path):
        data = symbol_context(path, symbol=symbol, depth=depth, verbose=False)
    return mcp_tool_result("symbol_context", arguments, data["text"], detail=detail, context=data)

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("edit_notebook"))
async def edit_notebook_tool(
    path: str, edits: list[dict], validate_code: bool = True,
    default_cell_type: str = "code", auto_feedback: bool = True,
    feedback_timeout: int = 10, feedback_safe: bool = True, detail: str = "summary",
    ctx: Context = None,
) -> ToolResult:
    "Apply deterministic notebook edit operations atomically."
    if not edits: raise ValueError("Pass at least one edit")
    root = await _mcp_workspace_root(ctx)
    path = _mcp_workspace_path(path, root)
    edits = _mcp_workspace_edit_paths(edits, root)
    arguments = dict(path=path, edits=edits, validate_code=validate_code, default_cell_type=default_cell_type, auto_feedback=auto_feedback, feedback_timeout=feedback_timeout, feedback_safe=feedback_safe, detail=detail, workspace_root=str(root) if root else None)
    result = edit_notebook(
        path, edits, validate_code=validate_code, default_cell_type=default_cell_type,
        auto_feedback=auto_feedback, feedback_timeout=feedback_timeout,
        feedback_safe=feedback_safe, detail=detail,
    )
    warnings = result.get("warnings", []) if isinstance(result, dict) else []
    full_output = result.get("text", str(result)) if isinstance(result, dict) else str(result)
    tool_result = mcp_tool_result(
        "edit_notebook", arguments, full_output, detail=detail, warnings=warnings,
        ok=bool(result.get("ok", False)) if isinstance(result, dict) else True,
        changed=result.get("changed") if isinstance(result, dict) else None,
        no_change=result.get("no_change") if isinstance(result, dict) else None,
        affected_cell_ids=result.get("affected_cell_ids", []) if isinstance(result, dict) else [],
        before_hash=result.get("before_hash") if isinstance(result, dict) else None,
        after_hash=result.get("after_hash") if isinstance(result, dict) else None,
        exported=result.get("exported") if isinstance(result, dict) else None,
    )
    if isinstance(result, dict): tool_result.structured_content["edit_notebook"] = result
    return tool_result

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("exec_nb"))
async def exec_nb_tool(
    path: str, dest: str | None = None, exc_stop: bool = False, up2id: int | str | None = None,
    chapter: str | None = None, timeout: int = 30, show_output: bool = True,
    verbose: bool = False, safe: bool = True, allow: str | None = None,
    ok_dests: str | None = None, cache_httpx: bool = False, cache_dir: str | None = None,
    cache_domains: str | None = None, allow_new: bool = False, check_only: bool = False,
    detail: str = "summary", ctx: Context = None,
) -> ToolResult:
    "Execute a notebook and return visible outputs/errors. Use check_only=True to avoid writing outputs."
    root = await _mcp_workspace_root(ctx)
    path = _mcp_workspace_path(path, root)
    dest = _mcp_workspace_path(dest, root) if dest else dest
    ok_dests = _mcp_workspace_paths(ok_dests, root) if ok_dests else ok_dests
    cache_dir = _mcp_workspace_path(cache_dir, root) if cache_dir else cache_dir
    arguments = dict(path=path, dest=dest, exc_stop=exc_stop, up2id=up2id, chapter=chapter, timeout=timeout, show_output=show_output, verbose=verbose, safe=safe, allow=allow, ok_dests=ok_dests, cache_httpx=cache_httpx, cache_dir=cache_dir, cache_domains=cache_domains, allow_new=allow_new, check_only=check_only, detail=detail, workspace_root=str(root) if root else None)
    if safe:
        full_output = capture_notebook_call(exec_nb, path, dest or path, **{k: v for k, v in arguments.items() if k not in {"detail", "workspace_root"}})
    else:
        full_output = _capture_exec_nb_cli_call(arguments)
    warnings = []
    if safe and "PermissionError: Audit:" in full_output:
        warnings.append(_warning(
            "safe-exec-audit-block",
            "Safe execution blocked an audited operation.",
            "For trusted notebooks, rerun with safe=False.",
        ))
    return mcp_tool_result("exec_nb", arguments, full_output, detail=detail, warnings=warnings)

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("diff_nb"))
async def _diff_nb_tool(path: str, ref_a: str | None = "HEAD", ref_b: str | None = None, adds: bool = True, changes: bool = True, dels: bool = False, cell_id: str | None = None, after_id: str | None = None, show_owner: bool = False, detail: str = "summary", ctx: Context = None) -> ToolResult:
    "Diff notebook code cells without expanding raw notebook JSON; optionally filter by cell id or map generated Python to its owner."
    root = await _mcp_workspace_root(ctx)
    path = _mcp_workspace_path(path, root)
    arguments = dict(path=path, ref_a=ref_a, ref_b=ref_b, adds=adds, changes=changes, dels=dels, cell_id=cell_id, after_id=after_id, show_owner=show_owner, detail=detail, workspace_root=str(root) if root else None)
    if show_owner and Path(path).suffix == ".py":
        return mcp_tool_result("diff_nb", arguments, _owner_output(path), detail=detail)
    full_output = capture_notebook_call(diff_nb, path, **{k: v for k, v in arguments.items() if k not in {"show_owner", "detail", "workspace_root"}})
    return mcp_tool_result("diff_nb", arguments, full_output, detail=detail)

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("execute_plan"))
async def execute_plan_tool(
    plan: str, notebook: str | None = None, scope: str = "notebook",
    notebooks: str | None = None, model: str | None = None, max_steps: int = 20,
    timeout: int = 30,
    detail: str = "summary", ctx: Context = None,
) -> ToolResult:
    "Run a bounded edit-interactive loop against one notebook or a project notebook set."
    root = await _mcp_workspace_root(ctx)
    notebook = _mcp_workspace_path(notebook, root) if notebook else notebook
    notebooks = _mcp_workspace_paths(notebooks, root) if notebooks else notebooks
    arguments = dict(plan=plan, notebook=notebook, scope=scope, notebooks=notebooks, model=model, max_steps=max_steps, timeout=timeout, detail=detail, workspace_root=str(root) if root else None)
    mode = "project" if scope == "project" or notebooks else "notebook"
    if mode == "project":
        project_args = dict(plan=plan, notebooks=notebooks, model=model, max_steps=max_steps, timeout=timeout, dry_run=False)
        full_output = capture_call(execute_project_plan, **project_args)
        return mcp_tool_result("execute_plan", arguments, full_output, detail=detail)
    if not notebook: raise ValueError("notebook is required when scope='notebook'")
    notebook_args = dict(notebook=notebook, plan=plan, model=model, max_steps=max_steps, timeout=timeout, dry_run=False)
    result = execute_plan(**notebook_args)
    full_output = plan_result_text(result)
    tool_result = mcp_tool_result("execute_plan", arguments, full_output, detail=detail)
    if isinstance(result, dict):
        tool_result.structured_content["history"] = result.get("history", [])
        tool_result.structured_content["summary"] = result.get("summary", "")
        tool_result.structured_content["execute_plan"] = result
    return tool_result

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("agent_workbench"))
async def agent_workbench_tool(
    goal: str, notebook: str | None = None, contract_file: str | None = None,
    execute: bool = False, max_steps: int = 8, timeout: int = 30,
    detail: str = "summary", ctx: Context = None,
) -> ToolResult:
    "Prepare or execute a taste-aware small-diff workbench run."
    root = await _mcp_workspace_root(ctx)
    notebook = _mcp_workspace_path(notebook, root) if notebook else notebook
    contract_file = _mcp_workspace_path(contract_file, root) if contract_file else contract_file
    arguments = dict(goal=goal, notebook=notebook, contract_file=contract_file, execute=execute, max_steps=max_steps, timeout=timeout, detail=detail, workspace_root=str(root) if root else None)
    result = agent_workbench_result(
        goal, notebook=notebook, contract_file=contract_file, execute=execute,
        max_steps=max_steps, timeout=timeout,
    )
    full_output = result.get("rendered_plan") or result.get("summary", "")
    tool_result = mcp_tool_result("agent_workbench", arguments, full_output, detail=detail)
    if isinstance(result, dict): tool_result.structured_content["agent_workbench"] = result
    return tool_result

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("symbol_graph"))
async def _symbol_graph_tool(path: str = "nbs", symbol: str = "", json_output: bool = False, detail: str = "summary", ctx: Context = None) -> ToolResult:
    "Show definitions, callers, and callees for one notebook symbol."
    root = await _mcp_workspace_root(ctx)
    path = _mcp_workspace_path(path, root)
    arguments = dict(path=path, symbol=symbol, json_output=json_output, detail=detail, workspace_root=str(root) if root else None)
    full_output = capture_call(symbol_graph, path=path, symbol=symbol, json_output=json_output)
    result = mcp_tool_result("symbol_graph", arguments, full_output, detail=detail)
    if symbol:
        result.structured_content["symbol_graph"] = symbol_graph_public_data(symbol_graph_data(path, symbol))
    return result

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("style_check"))
async def style_check_tool(
    path: str = ".", skip_folder_re: str | None = None, skip_path: str | None = None,
    strict: bool = False, delete_after_output: bool = False,
    max_output_chars: int = 12000, max_diagnostics: int = 200, fix: bool = False,
    changed_only: bool = False, ref_a: str | None = "HEAD", ref_b: str | None = None,
    detail: str = "summary", ctx: Context = None,
) -> ToolResult:
    "Print capped chkstyle output, notebook hygiene warnings, private symbol warnings, and global tool usage."
    root = await _mcp_workspace_root(ctx)
    path = _mcp_workspace_path(path, root)
    skip_path = _mcp_workspace_path(skip_path, root) if skip_path else skip_path
    arguments = dict(path=path, skip_folder_re=skip_folder_re, skip_path=skip_path, strict=strict, delete_after_output=delete_after_output, max_output_chars=max_output_chars, max_diagnostics=max_diagnostics, fix=fix, changed_only=changed_only, ref_a=ref_a, ref_b=ref_b, detail=detail, workspace_root=str(root) if root else None)
    style_output = capture_call(style_check, **{k: v for k, v in arguments.items() if k not in {"detail", "workspace_root"}})
    private_output = _private_symbol_report_text(path)
    full_output = "\n\n".join(chunk for chunk in [private_output, style_output] if chunk)
    report = style_report(
        path, max_output_chars=max_output_chars, max_diagnostics=max_diagnostics,
        changed_only=changed_only, ref_a=ref_a, ref_b=ref_b,
        skip_folder_re=skip_folder_re, skip_path=skip_path,
    )
    report["private_symbol_report"] = private_output
    return mcp_tool_result("style_check", arguments, full_output, max_output_chars=max_output_chars, detail=detail, style_report=report)

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("store_knowledge"))
async def _store_knowledge_tool(apply_regex: str, note: str, path: str | None = None, detail: str = "summary", ctx: Context = None) -> ToolResult:
    "Store or update one behaviour steering regex and note."
    root = await _mcp_workspace_root(ctx)
    path = _mcp_workspace_path(path or ".", root)
    arguments = dict(apply_regex=apply_regex, note=note, path=path, detail=detail, workspace_root=str(root) if root else None)
    full_output = capture_call(store_knowledge, apply_regex=apply_regex, note=note, path=path)
    return mcp_tool_result("store_knowledge", arguments, full_output, detail=detail)

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("add_behaviour_steering"))
async def _add_behaviour_steering_tool(regex: str, path: str | None = None, detail: str = "summary", ctx: Context = None) -> ToolResult:
    "Add a behaviour steering regex with a generic note."
    root = await _mcp_workspace_root(ctx)
    path = _mcp_workspace_path(path or ".", root)
    arguments = dict(regex=regex, path=path, detail=detail, workspace_root=str(root) if root else None)
    full_output = capture_call(add_behaviour_steering, regex=regex, path=path)
    return mcp_tool_result("add_behaviour_steering", arguments, full_output, detail=detail)

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("get_knowledge"))
async def _get_knowledge_tool(regex: str | None = None, path: str | None = None, detail: str = "summary", ctx: Context = None) -> ToolResult:
    "Return stored behaviour steering rules, optionally filtered by regex."
    root = await _mcp_workspace_root(ctx)
    path = _mcp_workspace_path(path or ".", root)
    arguments = dict(regex=regex, path=path, detail=detail, workspace_root=str(root) if root else None)
    full_output = capture_call(get_knowledge, regex=regex, path=path)
    return mcp_tool_result("get_knowledge", arguments, full_output, detail=detail)

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("reference_query"))
async def _reference_query_tool(
    query: str, top_k: int = 3, include_branch: bool = False,
    current_repo: str = ".", repos: str | None = None, path: str | None = None,
    kind: str | None = None, package: str | None = None, module: str | None = None, symbol: str | None = None,
    detail: str = "summary", ctx: Context = None,
) -> ToolResult:
    "Search globally indexed reference implementations."
    root = await _mcp_workspace_root(ctx)
    current_repo = _mcp_workspace_path(current_repo, root)
    path = _mcp_workspace_path(path, root) if path else path
    arguments = dict(
        query=query, top_k=top_k, include_branch=include_branch, current_repo=current_repo, repos=repos, path=path,
        kind=kind, package=package, module=module, symbol=symbol, detail=detail, workspace_root=str(root) if root else None,
    )
    result_data = reference_query(
        query, top_k=top_k, include_branch=include_branch, current_repo=current_repo, repos=repos, path=path,
        kind=kind, package=package, module=module, symbol=symbol,
    )
    full_output = json.dumps(result_data, indent=2, sort_keys=True)
    result = mcp_tool_result("reference_query", arguments, full_output, detail=detail)
    result.structured_content["reference_query"] = result_data
    return result

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("py2nb"))
async def py2nb_tool(
    path: str, nbs_path: str = "nbs", dest: str | None = None, recursive: bool = True,
    maxdepth: int | None = None, preserve_tree: bool = True, class_lines: int = 100,
    method_lines: int = 10, package: str | None = None, include: str | None = None,
    exclude: str | None = None, skip_init: bool = True, include_tests: bool = False,
    force: bool = True, detail: str = "summary", ctx: Context = None,
) -> ToolResult:
    "Convert one Python file or a folder of Python files into nbdev notebook source."
    root = await _mcp_workspace_root(ctx)
    path = _mcp_workspace_path(path, root)
    nbs_path = _mcp_workspace_path(nbs_path, root)
    dest = _mcp_workspace_path(dest, root) if dest else dest
    arguments = dict(path=path, nbs_path=nbs_path, dest=dest, recursive=recursive, maxdepth=maxdepth, preserve_tree=preserve_tree, class_lines=class_lines, method_lines=method_lines, package=package, include=include, exclude=exclude, skip_init=skip_init, include_tests=include_tests, force=force, detail=detail, workspace_root=str(root) if root else None)
    full_output = capture_call(py2nb, **{k: v for k, v in arguments.items() if k not in {"detail", "workspace_root"}}, dry_run=False)
    return mcp_tool_result("py2nb", arguments, full_output, detail=detail)

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("py2nbdev"))
async def _py2nbdev_tool(source: str, dest: str, package: str | None = None, nbs_path: str = "nbs", force: bool = False, run_validation: bool = True, detail: str = "summary", ctx: Context = None) -> ToolResult:
    "Create a pragmatic nbdev project from a pure-Python package."
    root = await _mcp_workspace_root(ctx)
    source = _mcp_workspace_path(source, root)
    dest = _mcp_workspace_path(dest, root)
    arguments = dict(source=source, dest=dest, package=package, nbs_path=nbs_path, force=force, run_validation=run_validation, detail=detail, workspace_root=str(root) if root else None)
    full_output = capture_call(py2nbdev, **{k: v for k, v in arguments.items() if k not in {"detail", "workspace_root"}}, dry_run=False)
    return mcp_tool_result("py2nbdev", arguments, full_output, detail=detail)

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("new_nbdev_notebook"))
async def new_nbdev_notebook_tool(
    name: str, default_exp: str | None = None, title: str | None = None,
    nbs_path: str = "nbs", force: bool = False, detail: str = "summary",
    ctx: Context = None,
) -> ToolResult:
    "Create a minimal nbdev source notebook and exported module."
    root = await _mcp_workspace_root(ctx)
    nbs_path = _mcp_workspace_path(nbs_path, root)
    arguments = dict(name=name, default_exp=default_exp, title=title, nbs_path=nbs_path, force=force, detail=detail, workspace_root=str(root) if root else None)
    full_output = capture_call(new_nbdev_notebook, **{k: v for k, v in arguments.items() if k not in {"detail", "workspace_root"}}, dry_run=False)
    return mcp_tool_result("new_nbdev_notebook", arguments, full_output, detail=detail)

In [ ]:

#| export
def create_mcp():
    "Return the public nbskill FastMCP server."
    return mcp

### Running the server

The CLI entry point only chooses the transport and starts FastMCP. Keeping startup separate from tool registration makes `create_mcp` easy to test without launching a long-running server.

In [ ]:
#| export
def main(
    transport: str = "stdio",  # MCP transport; stdio is what Codex/Claude use for local servers
    show_banner: bool = False,  # Show FastMCP startup banner
):
    "Run the nbskill MCP server."
    mcp.run(transport=transport, show_banner=show_banner)

### MCP transport-friendly edits

MCP clients should be able to send exact source directly. `source_lines` avoids JSON-string plans, temporary files, and CLI newline decoding entirely, which is useful when the source being written contains escaped notebook text. Unsafe notebook execution is still available through MCP, but it runs in a subprocess so signal-based timeouts stay in a main interpreter.

In [ ]:
mcp_demo = create_mcp()
mcp_tools = {tool.name: tool for tool in await mcp_demo.list_tools()}
assert "edits" in str(mcp_tools["edit_notebook"].parameters)
assert "auto_feedback" in str(mcp_tools["edit_notebook"].parameters)
assert "default_cell_type" in str(mcp_tools["edit_notebook"].parameters)
assert "edit_cell" not in mcp_tools
print("direct MCP schema exposes one structured edit tool")

In [ ]:
with write_demo_notebook("07_mcp_insert_cells.ipynb") as edit_nb:
    nb = new_nb([mk_cell('payload = "old"')])
    anchor_id = nb.cells[0].id
    _write_raw_nb(nb, edit_nb)
    result = await mcp_demo.call_tool("edit_notebook", {"path": str(edit_nb), "edits": [dict(op="insert_cells", anchor_id=anchor_id, cells=[dict(source_lines=["inserted = True"])])], "auto_feedback": False})
    assert result.structured_content["changed"] is True
    assert _read_raw_nb(edit_nb).cells[1].source == "inserted = True"
    print("direct MCP edit_notebook inserts structured cells")

In [ ]:
with write_demo_notebook("07_mcp_edit_notebook.ipynb") as edit_nb:
    nb = new_nb([mk_cell('payload = "old"\nother = 1')])
    edit_cell_id = nb.cells[0].id
    _write_raw_nb(nb, edit_nb)
    exact_source = 'payload = "line 1' + chr(92) + 'nline 2"'
    edit_result = await mcp_demo.call_tool("edit_notebook", {"path": str(edit_nb), "edits": [dict(op="replace_cell", cell_id=edit_cell_id, source_lines=[exact_source])], "auto_feedback": False})
    assert edit_result.structured_content["changed"] is True
    assert _read_raw_nb(edit_nb).cells[0].source == exact_source
    range_result = await mcp_demo.call_tool("edit_notebook", {"path": str(edit_nb), "edits": [dict(op="replace_lines", cell_id=edit_cell_id, start_line=1, end_line=1, replacement_lines=['payload = "range"'])], "auto_feedback": False})
    assert range_result.structured_content["changed"] is True
    assert _read_raw_nb(edit_nb).cells[0].source == 'payload = "range"'
    feedback_result = await mcp_demo.call_tool("edit_notebook", {"path": str(edit_nb), "edits": [dict(op="replace_cell", cell_id=edit_cell_id, source_lines=["print('mcp feedback')"])]})
    assert "Auto feedback" in feedback_result.structured_content["full_output"]
    assert "mcp feedback" in feedback_result.structured_content["full_output"]
    assert _read_raw_nb(edit_nb).cells[0].outputs == []
    print("direct MCP edit_notebook preserves escaped newlines and returns feedback")

In [ ]:
with write_demo_notebook("07_mcp_exec_nb.ipynb") as unsafe_nb:
    _write_raw_nb(new_nb([mk_cell("print('unsafe mcp')")]), unsafe_nb)
    exec_path = unsafe_nb.resolve()
    exec_result = await mcp_demo.call_tool("exec_nb", {"path": str(exec_path), "safe": False, "allow_new": True, "timeout": 5})
    assert "unsafe mcp" in exec_result.structured_content["full_output"]
    print("unsafe MCP exec_nb used subprocess without signal errors")

In [ ]:
mcp = create_mcp()

In [ ]:
tools = {tool.name: tool for tool in await mcp.list_tools()}

In [ ]:
assert {
    "healthcheck", "doctor", "project_context", "file_context", "chapter_context", "symbol_context",
    "edit_notebook", "exec_nb", "execute_plan", "agent_workbench", "symbol_graph",
    "style_check", "py2nb", "py2nbdev",
} <= set(tools)
assert not {"edit_cell", "edit_cell_range", "insert_cells", "apply_notebook_edits"} & set(tools)
assert not {"nb_overview", "nb_chapter", "nb_cell", "show_doc"} & set(tools)

In [ ]:
assert "write_nb" not in tools

In [ ]:
assert "update_cell" not in tools

In [ ]:
assert "batch_edit_nb" not in tools

In [ ]:
assert "execute_project_plan" not in tools

In [ ]:
assert "private_symbol_report" not in tools

In [ ]:
assert "py2nbs" not in tools

In [ ]:
assert "read_nb" not in tools

In [ ]:
assert "include_markdown" not in str(tools["file_context"].parameters)

In [ ]:
assert "show_ids" not in str(tools["project_context"].parameters)

In [ ]:
assert "show_ids" not in str(tools["chapter_context"].parameters)

In [ ]:
assert "show_ids" not in str(tools["symbol_context"].parameters)

In [ ]:
assert "include_re" in str(tools["file_context"].parameters)

In [ ]:
assert "verbose" not in str(tools["file_context"].parameters)

In [ ]:
assert "detail" in str(tools["file_context"].parameters)

In [ ]:
assert "show_owner" in str(tools["diff_nb"].parameters)

In [ ]:
assert "cell_id" in str(tools["diff_nb"].parameters)

In [ ]:
assert "after_id" in str(tools["diff_nb"].parameters)

In [ ]:
assert "check_only" in str(tools["exec_nb"].parameters)

In [ ]:
assert "scope" in str(tools["execute_plan"].parameters)

In [ ]:
assert "notebooks" in str(tools["execute_plan"].parameters)

In [ ]:
assert "edits" in str(tools["edit_notebook"].parameters)

In [ ]:
assert "default_cell_type" in str(tools["edit_notebook"].parameters)

In [ ]:
assert "auto_feedback" in str(tools["edit_notebook"].parameters)

In [ ]:
assert "feedback_timeout" in str(tools["edit_notebook"].parameters)

In [ ]:
assert "validate_code" in str(tools["edit_notebook"].parameters)

In [ ]:
assert "feedback_safe" in str(tools["edit_notebook"].parameters)

In [ ]:
assert "feedback_timeout" in str(tools["edit_notebook"].parameters)

In [ ]:
assert "edits" in str(tools["edit_notebook"].parameters)

In [ ]:
assert "max_output_chars" in str(tools["style_check"].parameters)

In [ ]:
assert "changed_only" in str(tools["style_check"].parameters)

In [ ]:
assert "scopes" in str(tools["doctor"].parameters)

In [ ]:
assert "json_output" in str(tools["symbol_graph"].parameters)

In [ ]:
assert "delete_after_outout" not in str(tools["style_check"].parameters)

In [ ]:
for name in ("edit_notebook", "execute_plan", "style_check", "py2nb", "py2nbdev"):
    assert "dry_run" not in str(tools[name].parameters)

In [ ]:
assert "Project context" in tools["project_context"].description

In [ ]:
assert "Implementation context" in tools["symbol_context"].description

In [ ]:
assert {"read", "notebook", "file"} <= set(tools["file_context"].tags)

In [ ]:
assert {"edit", "notebook", "cell"} <= set(tools["edit_notebook"].tags)

In [ ]:
assert {"edit", "notebook", "text"} <= set(tools["edit_notebook"].tags)

In [ ]:
assert "deterministic notebook edit operations" in tools["edit_notebook"].description

In [ ]:
assert "structured" in tools["edit_notebook"].description

In [ ]:
assert "atomically" in tools["edit_notebook"].description

In [ ]:
assert tools["execute_plan"].meta["feature"] == "agentic_planning"

In [ ]:
assert "Combined former execute_project_plan" in tools["execute_plan"].meta["combine_with"]

In [ ]:
assert tools["agent_workbench"].meta["feature"] == "agentic_planning"

In [ ]:
assert "execute" in str(tools["agent_workbench"].parameters)

In [ ]:
assert tools["py2nb"].meta["usefulness"] == "situational"

In [ ]:
assert "Combined former py2nbs" in tools["py2nb"].meta["combine_with"]

In [ ]:
assert "caller usage lines" in tools["symbol_graph"].description

In [ ]:
assert "CLI subprocess" in tools["exec_nb"].description

In [ ]:
graph_root = demo_path("07_mcp_symbol_graph")

In [ ]:
try:
    graph_root.mkdir()
    lib_nb = graph_root / "lib.ipynb"
    call_nb = graph_root / "call.ipynb"
    _write_raw_nb(new_nb([mk_cell("#| export\ndef thing():\n    return 1")]), lib_nb)
    _write_raw_nb(new_nb([mk_cell("from nbskill.lib import thing\nvalue = thing()")]), call_nb)
    symbol_result = await mcp.call_tool("symbol_graph", {"path": str(graph_root), "symbol": "thing", "json_output": True})
    assert symbol_result.structured_content["symbol_graph"]["caller_usages"][0]["line"] == "value = thing()"
    print("structured symbol_graph MCP usages:", len(symbol_result.structured_content["symbol_graph"]["caller_usages"]))
finally:
    remove_demo_path(graph_root)

In [ ]:
edit_root = demo_path("07_mcp_direct_update")

In [ ]:
async def _exercise_mcp_edit_tools(edit_root):
    edit_root.mkdir()
    edit_nb = edit_root / "edit.ipynb"
    nb = new_nb([mk_cell('payload = "old"\nvalue = 1')])
    cell_id = nb.cells[0].id
    _write_raw_nb(nb, edit_nb)
    exact_source = 'payload = "line 1' + chr(92) + 'nline 2"'
    edit_result = await mcp.call_tool("edit_notebook", {"path": str(edit_nb), "edits": [dict(op="replace_cell", cell_id=cell_id, source_lines=[exact_source])], "auto_feedback": False})
    assert edit_result.structured_content["changed"] is True
    assert _read_raw_nb(edit_nb).cells[0].source == exact_source

    range_result = await mcp.call_tool("edit_notebook", {"path": str(edit_nb), "edits": [dict(op="replace_lines", cell_id=cell_id, start_line=1, end_line=1, replacement_lines=['payload = "range"'])], "auto_feedback": False})
    assert range_result.structured_content["changed"] is True
    assert _read_raw_nb(edit_nb).cells[0].source == 'payload = "range"'

    insert_result = await mcp.call_tool("edit_notebook", {"path": str(edit_nb), "edits": [dict(op="insert_cells", anchor_id=cell_id, cells=[dict(cell_type="code", source_lines=["inserted = True"])])], "auto_feedback": False})
    assert insert_result.structured_content["changed"] is True
    inserted_id = _read_raw_nb(edit_nb).cells[1].id
    assert _read_raw_nb(edit_nb).cells[1].source == "inserted = True"

    move_result = await mcp.call_tool("edit_notebook", {"path": str(edit_nb), "edits": [dict(op="move_cells", cell_ids=[inserted_id], anchor_id=cell_id, where="before")], "auto_feedback": False})
    assert move_result.structured_content["changed"] is True
    assert _read_raw_nb(edit_nb).cells[0].id == inserted_id

    delete_result = await mcp.call_tool("edit_notebook", {"path": str(edit_nb), "edits": [dict(op="delete_cells", cell_ids=[inserted_id])], "auto_feedback": False})
    assert delete_result.structured_content["changed"] is True

In [ ]:
edit_root = demo_path("07_mcp_edit_tools")
try:
    await _exercise_mcp_edit_tools(edit_root)
    print("direct MCP edit tools used source_lines without JSON plan text")
finally:
    remove_demo_path(edit_root)

In [ ]:
edit_root = demo_path("07_mcp_unsafe_exec")
try:
    edit_root.mkdir()
    unsafe_nb = edit_root / "unsafe.ipynb"
    _write_raw_nb(new_nb([mk_cell("print('unsafe mcp')")]), unsafe_nb)
    exec_result = await mcp.call_tool("exec_nb", {"path": str(unsafe_nb.resolve()), "safe": False, "allow_new": True, "timeout": 5})
    assert "unsafe mcp" in exec_result.structured_content["full_output"]
    print("unsafe MCP exec_nb ran through CLI subprocess")
finally:
    remove_demo_path(edit_root)

In [ ]:
assert _doctor_scope_set("all") == {"error", "warning", "style"}

In [ ]:
doctor = _doctor_report(".", scopes="error,warning")

In [ ]:
assert doctor["style"] is None

In [ ]:
assert "errors" in doctor

In [ ]:
assert "warnings" in doctor

In [ ]:
assert "status" in doctor

In [ ]:
style_doctor = _doctor_report(".", scopes="style", max_output_chars=200, max_diagnostics=5)

In [ ]:
assert style_doctor["style"] is not None

In [ ]:
assert "chkstyle" in style_doctor["style"]

In [ ]:
redacted = mcp_tool_result(
    "edit_notebook",
    {"path": "nbs/example.ipynb", "edits": [dict(op="replace_cell", cell_id="abc123", source_lines=["x" * 1000])]},
    "ok",
)

In [ ]:
assert "x" * 200 not in redacted.structured_content["summary"]

In [ ]:
assert redacted.structured_content["call"]["arguments"]["edits"].startswith("<")

In [ ]:
debug = mcp_tool_result("file_context", {"path": "nbs/example.ipynb"}, "ok", detail="debug")

In [ ]:
assert debug.structured_content["debug"]["arguments"] == {"path": "nbs/example.ipynb"}

In [ ]:
calls = []

In [ ]:
old_doctor_warnings = _doctor_warnings

In [ ]:
def _capture_mcp_warning_scopes():
    try:
        def fake_doctor_warnings(path=".", scope_path=None, scope_cell_ids=None):
            cell_scope = None if scope_cell_ids is None else set(scope_cell_ids)
            calls.append((path, scope_path, cell_scope))
            return []

        globals()["_doctor_warnings"] = fake_doctor_warnings
        mcp_tool_result("healthcheck", {}, "ok")
        mcp_tool_result("project_context", {"path": "nbs"}, "Project context")
        mcp_tool_result("file_context", {"path": "nbs/file.ipynb"}, "File context")
        mcp_tool_result("chapter_context", {"path": "nbs/chapter.ipynb"}, "Cell id=ch1\nCell id=ch2")
        mcp_tool_result("symbol_context", {"path": "nbs/symbol.ipynb", "symbol": "thing"}, "Location: nbs/symbol.ipynb Cell id=def123: code")
        mcp_tool_result("diff_nb", {"path": "nbs/07_mcp.ipynb"}, "diff without cell ids")
        mcp_tool_result("diff_nb", {"path": "nbs/07_mcp.ipynb"}, "--- code cell abc123 ---\nchanged")
        mcp_tool_result("edit_notebook", {"path": "nbs/update_scope.ipynb", "edits": [dict(op="replace_cell", cell_id="upd123", source_lines=["x"])]}, "edit_notebook changed")
        mcp_tool_result("edit_notebook", {"path": "nbs/batch_a.ipynb", "edits": [dict(op="replace_cell", path="nbs/batch_b.ipynb", cell_id="b1", source_lines=["b"])]}, "edit_notebook changed")
    finally:
        globals()["_doctor_warnings"] = old_doctor_warnings

In [ ]:
_capture_mcp_warning_scopes()

In [ ]:
assert calls == [
    ("nbs/07_mcp.ipynb", "nbs/07_mcp.ipynb", set()),
    ("nbs/07_mcp.ipynb", "nbs/07_mcp.ipynb", {"abc123"}),
]


In [ ]:
source_calls = []

In [ ]:
root = git_root(".")

In [ ]:
old_generated_pairs = _generated_pairs_for_scope

In [ ]:
old_validation_problems = notebook_validation_problems

In [ ]:
old_style_warnings = _style_problem_warnings

In [ ]:
old_doc_warnings = _doc_script_warnings

In [ ]:
old_failure_data = _failure_data

In [ ]:
try:
    def fake_generated_pairs(root, path):
        source_calls.append(("generated", Path(path)))
        return []

    def fake_validation_problems(path):
        source_calls.append(("validation", Path(path)))
        return []

    def fake_style_warnings(path, root):
        source_calls.append(("style", Path(path)))
        return []

    def fake_doc_warnings(root):
        source_calls.append(("docs", Path(root)))
        return []

    def fake_failure_data():
        source_calls.append(("failure", None))
        return {"events": []}

    _generated_pairs_for_scope = fake_generated_pairs
    notebook_validation_problems = fake_validation_problems
    _style_problem_warnings = fake_style_warnings
    _doc_script_warnings = fake_doc_warnings
    _failure_data = fake_failure_data
    _doctor_warnings("nbs/cell.ipynb", scope_path="nbs/cell.ipynb")
    _doctor_warnings(".")
finally:
    _generated_pairs_for_scope = old_generated_pairs
    notebook_validation_problems = old_validation_problems
    _style_problem_warnings = old_style_warnings
    _doc_script_warnings = old_doc_warnings
    _failure_data = old_failure_data

In [ ]:
assert source_calls == [
    ("generated", root / "nbs/cell.ipynb"),
    ("validation", root / "nbs/cell.ipynb"),
    ("style", root / "nbs/cell.ipynb"),
    ("generated", root),
    ("validation", root),
    ("style", root),
    ("docs", root),
    ("failure", None),
]

In [ ]:
sample = [
    _warning("other", "other notebook", path="nbs/other.ipynb"),
    _warning("same_other_cell", "same notebook other cell", path="nbs/07_mcp.ipynb", cell_id="other-cell"),
    _warning("current", "current displayed cell", path="nbs/07_mcp.ipynb", cell_id="abc123"),
    _warning("notebook_export_missing", "current notebook", path="nbs/07_mcp.ipynb"),
]

In [ ]:
filtered = _filter_warnings_for_scope(sample, Path(".").resolve(), "nbs/07_mcp.ipynb", {"abc123"})

In [ ]:
assert [item["code"] for item in filtered] == ["current", "notebook_export_missing"]

In [ ]:
assert not _cell_export_relevant({"cell_type": "code", "source": ["print('demo')"]})

In [ ]:
assert _cell_export_relevant({"cell_type": "code", "source": ["#| export\ndef exported():\n    pass"]})

In [ ]:
print("notebook-only edits do not require generated-file warnings")

In [ ]:
module_path = Path("nbskill/mcp.py")

In [ ]:
if not module_path.exists(): module_path = Path("../nbskill/mcp.py")

In [ ]:
owner = generated_owner(module_path)

In [ ]:
assert owner and owner.name == "07_mcp.ipynb"

In [ ]:
assert "07_mcp.ipynb" in _owner_output(module_path)

In [ ]:
assert git_root("nbs/07_mcp.ipynb") == git_root(".")

In [ ]:
assert _rel_to_root("nbs/07_mcp.ipynb", git_root(".")) == "nbs/07_mcp.ipynb"

In [ ]:
root = demo_path("07_mcp_insert_lines")
try:
    root.mkdir()
    path = root / "example.ipynb"
    nb = new_nb([mk_cell("anchor = True")])
    anchor_id = nb.cells[0].id
    _write_raw_nb(nb, path)
    mcp = _mcp_mod.create_mcp()
    result = await mcp.call_tool(
        "edit_notebook",
        {
            "path": str(path),
            "edits": [dict(op="insert_cells", anchor_id=anchor_id, cells=[dict(cell_type="code", source_lines=['source = "line 1\\nline 2"'])])],
            "auto_feedback": False,
        },
    )
    assert result.structured_content["changed"] is True
    assert _read_raw_nb(path).cells[1].source == 'source = "line 1\\nline 2"'
finally:
    remove_demo_path(root)

In [ ]:
calls = {}

In [ ]:
project_calls = {}

In [ ]:
old_execute_plan = _mcp_mod.execute_plan

In [ ]:
old_execute_project_plan = _mcp_mod.execute_project_plan

In [ ]:
async def _exercise_agent_mcp_tools():
    try:
        def fake_execute_plan(**kwargs):
            calls.update(kwargs)
            return {"summary": "delegated summary", "history": [{"tool": "add_cell"}], "text": "delegated"}

        def fake_execute_project_plan(**kwargs):
            project_calls.update(kwargs)
            print("project delegated")
            return {"summary": "project summary"}

        _mcp_mod.execute_plan = fake_execute_plan
        _mcp_mod.execute_project_plan = fake_execute_project_plan
        server = _mcp_mod.create_mcp()
        result = await server.call_tool("execute_plan", {"notebook": "nbs/index.ipynb", "plan": "noop", "model": "fake", "max_steps": 1, "timeout": 2})
        assert calls == {"notebook": "nbs/index.ipynb", "plan": "noop", "model": "fake", "max_steps": 1, "timeout": 2, "dry_run": False}
        assert "delegated" in str(result)
        assert result.structured_content["summary"] == "delegated summary"
        assert result.structured_content["history"] == [{"tool": "add_cell"}]

        project = await server.call_tool("execute_plan", {"scope": "project", "notebooks": "nbs/01_read.ipynb,nbs/02_write.ipynb", "plan": "noop", "model": "fake", "max_steps": 1, "timeout": 2})
        assert project_calls == {"plan": "noop", "notebooks": "nbs/01_read.ipynb,nbs/02_write.ipynb", "model": "fake", "max_steps": 1, "timeout": 2, "dry_run": False}
        assert "project delegated" in str(project)

        project_root = Path.cwd().parent if Path.cwd().name == "nbs" else Path.cwd()
        token = _mcp_mod._in_call_parse.set(True)
        try:
            workbench = await server.call_tool("agent_workbench", {"goal": "touch nothing", "notebook": str(project_root / "nbs/11_agent_workbench.ipynb")})
        finally:
            _mcp_mod._in_call_parse.reset(token)
        assert "Agent workbench task" in str(workbench)
        assert workbench.structured_content["agent_workbench"]["summary"] == "agent_workbench prepared execution context"
    finally:
        _mcp_mod.execute_plan = old_execute_plan
        _mcp_mod.execute_project_plan = old_execute_project_plan

In [ ]:
await _exercise_agent_mcp_tools()

In [ ]:
with write_demo_notebook("07_mcp_sample.ipynb") as path:
    _example_write_nb(
        str(path),
        "%%code\n"
        "#| default_exp sample\n"
        "def sample():\n"
        "    return 'ok'",
        replace=True,
    )
    text = capture_notebook_call(_example_file_context, path, path=str(path))
    assert "def sample():" in text
    print("captured file context lines:", len(text.splitlines()))

In [ ]:
assert as_text(None) == ""
assert as_text({"ok": True}) == "{'ok': True}"
assert capture_call(lambda: "returned") == "returned"

In [ ]:
def _prints_and_returns():
    print("printed")
    return "returned"

In [ ]:
assert capture_call(_prints_and_returns) == "printed"

In [ ]:
def _prints_and_exits():
    print("before exit")
    raise SystemExit(7)

In [ ]:
try:
    capture_call(_prints_and_exits)
except RuntimeError as exc:
    assert "before exit" in str(exc)
    assert "SystemExit: 7" in str(exc)
else:
    raise AssertionError("SystemExit should be converted to RuntimeError for MCP tools")

In [ ]:
import time

In [ ]:
original_stdout = sys.stdout
outputs = []
errors = []
entered = threading.Event()

In [ ]:
def _slow_print(label, delay, signal=None):
    def inner():
        if signal is not None: signal.set()
        time.sleep(delay)
        print(label)
    return inner

In [ ]:
def _capture_worker(label, delay, signal=None):
    try:
        outputs.append(capture_call(_slow_print(label, delay, signal=signal)))
    except BaseException as exc:
        errors.append(exc)

In [ ]:
cell_stdout = sys.stdout
threads = [
    threading.Thread(target=_capture_worker, args=("first", 0.03, entered)),
    threading.Thread(target=_capture_worker, args=("second", 0.01)),
]
threads[0].start()
assert entered.wait(1)
threads[1].start()
for thread in threads: thread.join()

assert errors == []
assert sorted(outputs) == ["first", "second"]
assert sys.stdout is cell_stdout